# Bulgarian FastSpeech2 — run **MFA locally** (Windows + conda)

Colab keeps killing the runtime mid-MFA. MFA is **CPU-only** (Kaldi/GMM-HMM — no GPU), so run it here on your 16-core machine where nothing times out, then carry the alignments back to Colab for the **GPU training** only.

**Data source:** `RealData/merged_dataset/` (already unzipped: 44k wavs + `manifest.csv`). Nothing is read from `filelists/`. All generated intermediates go to `mfa_work/`.

**Outputs** (in `mfa_local_out/`) — two zips you upload to `MyDrive/fs2_bg_real/`:
- `raw_data_Bulgarian.zip` — resampled 22.05 kHz wavs + `.lab`
- `TextGrid_Bulgarian.zip` — the MFA alignments

On Colab you then run **section 10b (Recovery)** -> Preprocess -> Train. **MFA never runs on Colab again.**

**Kernel:** pick your base Anaconda Python. All heavy work runs in a dedicated `mfa` conda env via `conda run`, so the notebook kernel itself only needs the standard library. Every cell is **resumable** — re-run it and it skips finished work.

## 1. One-time setup — create the `mfa` conda env

Run these **once** in the VSCode terminal (PowerShell), *not* in the notebook:

```powershell
conda create -n mfa -c conda-forge montreal-forced-aligner -y
conda install -n mfa -c conda-forge librosa tqdm pyyaml -y
```

First line = MFA + Kaldi (a few minutes). Second = what `prepare_align.py` needs to resample. The next cell verifies it worked.

In [ ]:
import os, sys, re, glob, wave, time, random, shutil, threading, subprocess

# --- run from the REPO ROOT no matter where this notebook lives (it sits in local/) ---
# Every path below is relative to the repo root, no leading "../". We walk up to the repo
# marker and chdir there, so it works whether Jupyter starts in local/ or the root.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "config", "Bulgarian", "preprocess.yaml")):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        raise RuntimeError("repo root not found - open this notebook inside the FastSpeech2 repo")
    _root = _parent
os.chdir(_root)
print("repo root  :", os.getcwd())


# ---------- conda env that has MFA + librosa ----------
ENV   = "mfa"
CONDA = shutil.which("conda") or os.path.join(sys.prefix, "Scripts", "conda.exe")
if not os.path.exists(CONDA):
    CONDA = "conda"   # fall back to PATH

# ---------- data (already unzipped locally; the ONLY data source) ----------
WAVS_DIR     = "RealData/merged_dataset/wavs"
SRC_MANIFEST = "RealData/merged_dataset/manifest.csv"
LEXICON      = "lexicon/dictionary_full.txt"

# ---------- how much data ----------
USE_ALL         = True     # align the whole corpus. Set False to cap at TRAIN_HOURS.
TRAIN_HOURS     = 60.0     # only used when USE_ALL = False
MFA_TRAIN_HOURS = 20.0     # subset used ONLY to train the aligner

# ---------- compute ----------
JOBS = 6   # 16 cores but ~14 GB RAM, so RAM is the limit. 6 is safe.
           # lower to 4 if MFA crashes/OOMs; raise to 8 if RAM stays comfortable.

# ---------- outputs / local work area ----------
OUT_BACKUP = "mfa_local_out"   # the zips you upload to Drive
WORK       = "mfa_work"        # generated intermediates (manifest, local config) - safe to delete
os.makedirs(OUT_BACKUP, exist_ok=True)
os.makedirs(WORK, exist_ok=True)

def run_stream(args):
    """Run a command, stream its output live, print a 60 s heartbeat + elapsed."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True); print("-"*70, flush=True)
    t0 = time.time(); done = threading.Event()
    def hb():
        while not done.wait(60):
            print(f"  ... still running - {(time.time()-t0)/60:.1f} min", flush=True)
    threading.Thread(target=hb, daemon=True).start()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         bufsize=1, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait(); done.set()
    print("-"*70, flush=True)
    print(f"exit {p.returncode} after {(time.time()-t0)/60:.1f} min", flush=True)
    return p.returncode

def mfa(*a):
    return run_stream([CONDA, "run", "--no-capture-output", "-n", ENV, "mfa", *a])

def dur_wav(p):
    try:
        with wave.open(p) as w: return w.getnframes()/w.getframerate()
    except Exception:
        return None

# ---- verify ----
print("conda :", CONDA)
print("cores :", os.cpu_count(), "| JOBS =", JOBS)
print("wavs  :", os.path.isdir(WAVS_DIR), "| manifest:", os.path.exists(SRC_MANIFEST), "| lexicon:", os.path.exists(LEXICON))
v = subprocess.run([CONDA, "run", "-n", ENV, "mfa", "version"], capture_output=True, text=True)
print("mfa version:", (v.stdout or v.stderr).strip() or "NOT FOUND - create the env first (cell above)")

## 2. Build the working manifest *(from RealData)*
Reads `RealData/merged_dataset/manifest.csv`, rewrites each wav path to the real local file, optionally subsets by hours, and writes everything into `mfa_work/`. Also writes a local copy of `preprocess.yaml` so the tracked config stays clean.

In [ ]:
rows = []
with open(SRC_MANIFEST, encoding="utf-8") as f:
    for line in f:
        a = line.rstrip("\n").split("|")
        if len(a) < 3:
            continue
        uid, text = a[0], a[2]
        wav = WAVS_DIR + "/" + uid + ".wav"
        if os.path.exists(wav):
            rows.append((uid, wav, text))
print("clips with audio:", len(rows))

random.seed(42); random.shuffle(rows)
if USE_ALL:
    sel = rows
    print("USE_ALL -> aligning the whole corpus")
else:
    sel, tot = [], 0.0
    for r in rows:
        d = dur_wav(r[1])
        if d is None:
            continue
        sel.append(r); tot += d
        if tot >= TRAIN_HOURS*3600:
            break
    print(f"subset: {len(sel)} clips ~ {tot/3600:.1f} h")

LOCAL_MANIFEST = f"{WORK}/realdata_local.csv"
with open(LOCAL_MANIFEST, "w", encoding="utf-8") as f:
    for uid, wav, text in sel:
        f.write(f"{uid}|{wav}|{text}\n")
print("wrote", LOCAL_MANIFEST, "->", len(sel), "clips")

# local config copy (don't mutate the tracked preprocess.yaml)
PRE_LOCAL = f"{WORK}/preprocess.local.yaml"
s = open("config/Bulgarian/preprocess.yaml", encoding="utf-8").read()
s = re.sub(r'(manifest_path:\s*")[^"]*(")', r'\g<1>' + LOCAL_MANIFEST + r'\g<2>', s, count=1)
s = re.sub(r'(corpus_path:\s*")[^"]*(")',  r'\g<1>.\g<2>', s, count=1)
open(PRE_LOCAL, "w", encoding="utf-8").write(s)
print("wrote", PRE_LOCAL)

## 3. prepare_align — resample to 22.05 kHz + write `.wav`/`.lab`
This is the slowest **serial** step (tens of minutes for the full corpus). It writes `raw_data/Bulgarian/Bulgarian/`. Skips if already populated.

In [ ]:
SRC = "raw_data/Bulgarian/Bulgarian"
have = len(glob.glob(f"{SRC}/*.wav"))
if have >= len(sel):
    print(f"raw_data already has {have} wavs (>= {len(sel)}) -> skip prepare_align")
else:
    run_stream([CONDA, "run", "--no-capture-output", "-n", ENV,
                "python", "prepare_align.py", PRE_LOCAL])
print("raw_data wavs:", len(glob.glob(f"{SRC}/*.wav")))

## 4. Pseudo-speaker corpus (the speed fix)
Regroup clips by `book__pista` (~150 "speakers") so MFA uses all cores instead of running single-threaded.

In [ ]:
SRC = "raw_data/Bulgarian/Bulgarian"
MFA_CORPUS = "mfa_corpus"
shutil.rmtree(MFA_CORPUS, ignore_errors=True)
n, spks = 0, set()
for wav in glob.glob(f"{SRC}/*.wav"):
    uid = os.path.splitext(os.path.basename(wav))[0]
    spk = "__".join(uid.split("__")[:2]) or "spk0"
    d = os.path.join(MFA_CORPUS, spk); os.makedirs(d, exist_ok=True); spks.add(spk)
    for f in (wav, wav[:-4] + ".lab"):
        if os.path.exists(f):
            dst = os.path.join(d, os.path.basename(f))
            try: os.link(f, dst)
            except OSError: shutil.copy(f, dst)
    n += 1
print(f"pseudo-speaker corpus: {n:,} clips / {len(spks)} speakers")

## 5. *(optional)* validate
Quality check — flags out-of-vocabulary words / unreadable audio. Skip to save time.

In [ ]:
mfa("validate", "mfa_corpus", LEXICON, "-j", JOBS, "--clean")

## 6. Train the grapheme aligner *(checkpoint)*
Trains on `MFA_TRAIN_HOURS` of speakers and copies the model to `mfa_local_out/`. Skips if the model is already there.

In [ ]:
MODEL = "bg_acoustic_model.zip"
if os.path.exists(os.path.join(OUT_BACKUP, MODEL)):
    shutil.copy(os.path.join(OUT_BACKUP, MODEL), MODEL)
    print("model already in", OUT_BACKUP, "-> skip training")
else:
    def spk_hours(spk):
        return sum((dur_wav(w) or 0) for w in glob.glob(f"mfa_corpus/{spk}/*.wav"))
    random.seed(0)
    spk_all = sorted(os.listdir("mfa_corpus")); random.shuffle(spk_all)
    keep, tot = [], 0.0
    for spk in spk_all:
        keep.append(spk); tot += spk_hours(spk)
        if tot >= MFA_TRAIN_HOURS*3600:
            break
    print(f"[select] {len(keep)} speakers ~ {tot/3600:.1f} h to train the aligner", flush=True)
    shutil.rmtree("mfa_train", ignore_errors=True)
    for spk in keep:
        os.makedirs(f"mfa_train/{spk}", exist_ok=True)
        for f in glob.glob(f"mfa_corpus/{spk}/*"):
            try: os.link(f, f"mfa_train/{spk}/{os.path.basename(f)}")
            except OSError: shutil.copy(f, f"mfa_train/{spk}/{os.path.basename(f)}")
    run_stream([CONDA, "run", "--no-capture-output", "-n", ENV,
                "mfa", "train", "mfa_train", LEXICON, MODEL, "-j", JOBS, "--clean"])
    assert os.path.exists(MODEL), "mfa train failed - see the log above"
    shutil.copy(MODEL, OUT_BACKUP)
    print("[backup] model ->", os.path.join(OUT_BACKUP, MODEL))

## 7. Align book-by-book *(resumable)*
Aligns one book at a time; each finished book is zipped into `mfa_local_out/textgrids/`. Re-running skips finished books, so even if you stop the kernel you only lose the current book.

In [ ]:
from collections import defaultdict
MODEL = "bg_acoustic_model.zip"
TG_OUT = "preprocessed_data/Bulgarian/TextGrid/Bulgarian"
os.makedirs(TG_OUT, exist_ok=True)
os.makedirs(f"{OUT_BACKUP}/textgrids", exist_ok=True)

books = defaultdict(list)
for spk in sorted(os.listdir("mfa_corpus")):
    books[spk.split("__")[0]].append(spk)

for book, spk_list in sorted(books.items()):
    bzip = f"{OUT_BACKUP}/textgrids/{book}.zip"
    if os.path.exists(bzip):
        shutil.unpack_archive(bzip, TG_OUT)
        print(f"[skip] {book} (already aligned)"); continue
    shutil.rmtree("mfa_chunk", ignore_errors=True); shutil.rmtree("mfa_chunk_out", ignore_errors=True)
    for spk in spk_list:
        os.makedirs(f"mfa_chunk/{spk}", exist_ok=True)
        for f in glob.glob(f"mfa_corpus/{spk}/*"):
            try: os.link(f, f"mfa_chunk/{spk}/{os.path.basename(f)}")
            except OSError: shutil.copy(f, f"mfa_chunk/{spk}/{os.path.basename(f)}")
    print(f"[align] {book}: {len(spk_list)} pistas", flush=True)
    run_stream([CONDA, "run", "--no-capture-output", "-n", ENV,
                "mfa", "align", "mfa_chunk", LEXICON, MODEL, "mfa_chunk_out", "--clean", "-j", JOBS])
    tgs = glob.glob("mfa_chunk_out/**/*.TextGrid", recursive=True)
    shutil.rmtree("_bk", ignore_errors=True); os.makedirs("_bk")
    for tg in tgs:
        shutil.copy(tg, TG_OUT); shutil.copy(tg, "_bk")
    shutil.make_archive(f"{OUT_BACKUP}/textgrids/{book}", "zip", "_bk")
    print(f"[done] {book}: {len(tgs)} TextGrids", flush=True)

print("Total TextGrids:", len(glob.glob(f"{TG_OUT}/*.TextGrid")))

## 8. Package the two zips for Colab

In [ ]:
shutil.make_archive(f"{OUT_BACKUP}/raw_data_Bulgarian", "zip", "raw_data", "Bulgarian")
shutil.make_archive(f"{OUT_BACKUP}/TextGrid_Bulgarian", "zip", "preprocessed_data/Bulgarian", "TextGrid")
print("ready to upload to MyDrive/fs2_bg_real:")
for f in sorted(os.listdir(OUT_BACKUP)):
    p = os.path.join(OUT_BACKUP, f)
    if os.path.isfile(p):
        print(f"  {f:30s} {os.path.getsize(p)/1e6:9.1f} MB")

## Done — finish on Colab (GPU)

1. Upload the two zips from `mfa_local_out/` to **`MyDrive/fs2_bg_real/`**.
2. On Colab open the REAL training notebook and run sections **1, 2, 3, 4, 5**, then **7** (vocoder), then the **Recovery cell (section 10b)** — it restores `raw_data` + TextGrids and **skips MFA entirely**. (The recovery cell now treats `filelists.zip` as optional, so the two zips are enough.)
3. Then **Section 11 (Preprocess)** -> **12 (symlink output)** -> **13 Option A (train from zero)**.

### Troubleshooting MFA on Windows
- **Out of memory / crash:** lower `JOBS` to 4 (RAM, not cores, is the limit at ~14 GB).
- **Database / server error:** delete MFA's temp dir and re-run — in the terminal: `Remove-Item -Recurse -Force $env:USERPROFILE\Documents\MFA`. `--clean` already resets per-corpus state.
- **Hardlink error across drives:** the code auto-falls back to copying, so it's safe — just uses more disk.